### Hyperparameter tuning on ANN

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential #type: ignore
from tensorflow.keras.layers import Dense #type: ignore
from tensorflow.keras.callbacks import EarlyStopping #type: ignore
import pickle

In [2]:
# Load the dataset
df = pd.read_csv("bank.csv")

In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Preprocessing

In [4]:
# Drop irrelevant columns
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

In [5]:
# Encode the categorical variables
label_encoder_gender = LabelEncoder()
df["Gender"] = label_encoder_gender.fit_transform(df["Gender"])

In [6]:
onehot_encoder_geography = OneHotEncoder()
geography_encoder = onehot_encoder_geography.fit_transform(df[["Geography"]])
geography_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [7]:
onehot_encoder_geography.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
geo_encoded_df = pd.DataFrame(geography_encoder.toarray(), columns=onehot_encoder_geography.get_feature_names_out(["Geography"])) #type: ignore

In [9]:
# Combine all the columns with original data
df = pd.concat([df.drop("Geography", axis=1), geo_encoded_df], axis=1)

In [10]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
# Divide the dataset into dependent and independent features
X = df.drop("Exited", axis=1)
y = df["Exited"]

In [12]:
# Split the data in train and test dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Scale the feaures
scaler = StandardScaler()

In [14]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [15]:
# Save the encoders and scaler 
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)
    
with open("onehot_encoding_geography.pkl", "wb") as file:
    pickle.dump(onehot_encoder_geography, file)
    
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

In [ ]:
# Define a function to create the model and try different parameters
def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation="relu", input_shape=(X_train.shape[1],))) #type: ignore
    
    for _ in range(layers-1):
        model.add(Dense(neurons, activation="relu"))
        
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [27]:
# Create a Keras classifier
model = KerasClassifier(
    model=create_model,
    neurons=32,    # ← must be declared here
    layers=1,      # ← must be declared here
    epochs=50,
    batch_size=10,
    verbose=0
)

In [28]:
# Grid Search parameters
params = {
    "neurons" : [16, 32, 64, 128],
    "layers" : [1, 2, 3],
    "epochs" : [50, 100]
}

In [31]:
# Perform Grid SearchCV
grid = GridSearchCV(estimator=model, param_grid=params, n_jobs=-1, cv=3, verbose=1)

In [30]:
grid_result = grid.fit(X_train, y_train)

In [32]:
# Print the best parameters
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

Best: 0.8592489798490117 using {'epochs': 50, 'layers': 1, 'neurons': 16}
